In [1]:
import pandas as pd
import numpy as np
from collections import Counter

df = pd.read_csv('../data/train_features.csv')
print(f"Shape: {df.shape}")
print(f"Fraud: {Counter(df['isFraud'])}")

Shape: (10000, 193)
Fraud: Counter({0: 9735, 1: 265})


In [2]:
# Sắp xếp theo thời gian — RẤT QUAN TRỌNG
df = df.sort_values('TransactionDT').reset_index(drop=True)

# 80% đầu = train, 20% cuối = test
split_idx = int(len(df) * 0.8)

train = df.iloc[:split_idx]
test  = df.iloc[split_idx:]

print(f"Train: {train.shape} | Fraud: {train['isFraud'].sum()}")
print(f"Test:  {test.shape}  | Fraud: {test['isFraud'].sum()}")
print(f"\nTrain time range: {train['TransactionDT'].min()} → {train['TransactionDT'].max()}")
print(f"Test  time range: {test['TransactionDT'].min()} → {test['TransactionDT'].max()}")
print(f"\n✅ Test hoàn toàn SAU train về thời gian: {train['TransactionDT'].max() <= test['TransactionDT'].min()}")

Train: (8000, 193) | Fraud: 188
Test:  (2000, 193)  | Fraud: 77

Train time range: 86400 → 246623
Test  time range: 246624 → 313121

✅ Test hoàn toàn SAU train về thời gian: True


In [3]:
from imblearn.over_sampling import SMOTE

X_train = train.drop(columns=['isFraud'])
y_train = train['isFraud']
X_test  = test.drop(columns=['isFraud'])
y_test  = test['isFraud']

print(f"Train TRƯỚC SMOTE: {Counter(y_train)}")

smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f"Train SAU SMOTE:   {Counter(y_train_sm)}")
print(f"\n⚠️  Test KHÔNG dùng SMOTE: {Counter(y_test)}")

Train TRƯỚC SMOTE: Counter({0: 7812, 1: 188})
Train SAU SMOTE:   Counter({0: 7812, 1: 7812})

⚠️  Test KHÔNG dùng SMOTE: Counter({0: 1923, 1: 77})


In [4]:
# Lưu train
train_final = pd.DataFrame(X_train_sm, columns=X_train.columns)
train_final['isFraud'] = y_train_sm.values
train_final.to_csv('../data/train_final.csv', index=False)

# Lưu test
test_final = pd.DataFrame(X_test, columns=X_test.columns)
test_final['isFraud'] = y_test.values
test_final.to_csv('../data/test_final.csv', index=False)

print(f"✅ Train final: {train_final.shape}")
print(f"✅ Test final:  {test_final.shape}")
print("Saved!")

C:\Users\minh4\AppData\Local\Temp\ipykernel_4416\3856495662.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_final['isFraud'] = y_train_sm.values


✅ Train final: (15624, 193)
✅ Test final:  (2000, 193)
Saved!
